# Deployment Runbook — Dev Environment (Steps 3–4)

**Framework:** File Validation & Routing (`fvr`) • **Purpose:** mechanical bring-up of the dev stack — MongoDB (metadata) + FastAPI upload API + validation/routing worker (bytes on UC volumes) • **Status (2026-11-09):** all artifacts authored and reviewed; **MongoDB is NOT yet provisioned** (no Mongo connection exists in this workspace — verified against Unity Catalog connections), so this runbook is executed on infrastructure outside Databricks: an Atlas cluster and a small container host (Docker Compose or k8s) for the two services.

## 1. Already done (do not redo)

| Item | State |
| --- | --- |
| UC volumes `mdd_platform.fvr.{staging,intake,quarantine}` | ✅ created, roots seeded in `volume_map`, smoke-tested |
| Registry + operational DDL & seeds (Steps 1–2, mongosh) | ✅ authored, re-runnable, real schema contracts |
| Upload API code (Step 3) | ✅ authored + reviewed (F1–F4 fixed) |
| Validation/routing worker code (Step 4) | ✅ authored + reviewed (V1–V6 fixed) |
| R2 fixtures (`fvr_test_files/`) + dry-run matrix | ✅ generated, headers verified; **pre-R2 simulation 11/11 PASS on real volumes** (see R2 notebook execution log) |

## 2. Bring-up sequence (each step is idempotent — safe to re-run)

| # | Step | Where | Verify |
| --- | --- | --- | --- |
| 1 | **Atlas cluster** — ✅ cluster **FileValidator** provisioned 2026-11-10 (project `6af2ab427c4f5e1984b5eb31`); DB user `kishoreav_db_user` (⚠️ rotate this password after bring-up — it transited chat; credentials belong in the `fvr_dev` secret scope, never in code or chat) | Atlas console | pending: SRV hostname for the connection string + IP access list entries (Databricks serverless egress ranges, or dev shortcut `0.0.0.0/0`) |
| 2 | **Apply Step 1** (registry DDL + seeds) then **Step 2** (operational DDL + index guards) | `mongosh` from any allowlisted host | both notebooks' verification cells: 7 collections seeded 1/5/5/8/3/7, all 8 ops collections `validationAction: error`, 0 TTL indexes |
| 3 | **Service principal** `svc-fvr-dev` in Databricks; generate OAuth secret (M2M) | Databricks admin | token exchange curl returns an access token |
| 4 | **Grants** — ✅ **applied 2026-11-10**: `READ VOLUME` + `WRITE VOLUME` on `mdd_platform.fvr.{staging,intake,quarantine}` for `svc-fvr-dev` (application ID `0b30378e-d67d-4b33-87f5-356e83631494`, principal 77886323905562). Note: this metastore's privilege version rejects `READ FILES`/`WRITE FILES` on volumes — volume privileges are `READ VOLUME`/`WRITE VOLUME`. Prod: split into two SPs (API: staging RW; worker: staging read + intake/quarantine write) | UC permissions | each volume shows both privileges for the SP |
| 5 | **Deploy API** — `uvicorn fvr_api.main:app` behind gunicorn (2 pods), env per cell 3, `/healthz` + `/metrics` | container host | `curl $API/healthz` → 200; startup cross-check (volume_map root vs settings) passes |
| 6 | **Deploy worker** — `python -m worker.main` (1 pod, dev), same env + `WORKER_ID`; `/metrics` | container host | `event_backlog` metric appears; no claim errors in logs |
| 7 | **Step-3 smoke** — manifest → stream → complete (cell 6 of the Step-3 notebook) against dev | any host with curl | `COMPLETED`, `file_registry` RECEIVED, exactly one `FILE_ARRIVED`, staging copy present |
| 8 | **Step-4 smoke** — watch the file flow `RECEIVED → ROUTED_INTAKE`, 8 PASS rows, staging dir gone | mongosh | worker pass criteria cell in Step-4 notebook |
| 9 | **R2 full execution** — cases 1–12 (incl. the kill -9 chaos case) + locust S1–S5, record verdict | locust host | R2 notebook sign-off table |

## 3. Environment & secrets (single source)

| Variable | Value (dev) | Notes |
| --- | --- | --- |
| `DATABRICKS_HOST` | `https://dbc-16f873ab-edf2.cloud.databricks.com` | this workspace |
| `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET` | `0b30378e-d67d-4b33-87f5-356e83631494` (svc-fvr-dev OAuth M2M) + secret | **remaining manual step:** generate the OAuth secret (account console → service principal → OAuth secrets, or `databricks secrets create`-equivalent account CLI) and store it in your secret manager — **never** in the repo; dev fallback: a PAT in `DATABRICKS_TOKEN` |
| `MONGO_URI` | `mongodb+srv://fvr-api:<pw>@<cluster>…/fvr?retryWrites=true` | Atlas; user has readWrite on `fvr` only |
| `STAGING_ROOT` | `/Volumes/mdd_platform/fvr/staging` | cross-checked against `volume_map` at startup — mismatch = 503 fail-fast |

**Rollback:** services are stateless — `docker compose down` leaves only Mongo + volume bytes; a bad deploy is rolled back by redeploying the previous image tag. Mongo DDL is guarded/idempotent (Steps 1–2 re-runnable); a hard reset for dev = drop the `fvr` database and re-run Steps 1–2 — **never** drop the UC volumes (they hold intake bytes; re-create only per the Purge Job rules).

**Common failure modes:** `401/403` on volume PUT → SP missing `READ VOLUME`/`WRITE VOLUME` (this metastore rejects the `FILES` variants) or wrong host; Mongo `TransactionalNotSupported` → single-node Atlas without replica set (use M10+ or a replica-set M0); startup 503 → `volume_map` roots vs `STAGING_ROOT` drift; worker claims nothing → check the V1 `$or` claim filter (a regression reintroducing a bare `$lt` starves all fresh events).

## 4. After dev is up

* **R2 full run** (step 9) closes Phase-1 validation; record the verdict + issues log in the R2 notebook.
* **Prod deltas** (not dev): two separate SPs (API: staging only; worker: staging-read + intake/quarantine-write); HA Atlas; gunicorn workers sized from S3 numbers; alerts wired to the `notifications` seeds; environment-specific volumes (open item #5 of the Storage Addendum) — `volume_map` is env-scoped data, not code.

In [0]:
# Databricks notebook source
# Atlas connectivity smoke test — Databricks compute → FileValidator (runbook step 1 verification)
# Prereqs: secrets `atlas_user`, `atlas_password`, `atlas_srv_host` in scope `fvr_dev`, AND an
#          Atlas IP access list entry that covers this compute's egress (dev shortcut: 0.0.0.0/0).

%pip install pymongo -q
dbutils.library.restartPython()

import pymongo

user = dbutils.secrets.get("fvr_dev", "atlas_user")
pw   = dbutils.secrets.get("fvr_dev", "atlas_password")            # rotated value — lives only in the secret scope
host = dbutils.secrets.get("fvr_dev", "atlas_srv_host")            # e.g. filevalidator.xxxxx.mongodb.net
uri  = f"mongodb+srv://{user}:{pw}@{host}/fvr?retryWrites=true&w=majority"  # never printed

client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=15000)

info = client.admin.command("hello")
print("cluster setName:", info.get("setName"), "| replica hosts:", len(info.get("hosts", [])))
print("databases visible:", client.list_database_names())

client["fvr"].command("ping")
print("ping OK — FileValidator reachable from Databricks compute; Steps 1–2 can be applied from here.")

In [0]:
# Databricks notebook source
# Applies Build Steps 1-2 (7 registry + 8 operational collections) to Atlas FileValidator via
# pymongo, reading credentials from secret scope fvr_dev. Source of truth: the Step-1/Step-2
# mongosh notebooks (this cell is a faithful pymongo translation — including the legacy
# MONGO_TO_* direction enum, which matches the Step-2 DDL and the Step-4 worker).
# Idempotent: guarded create, create_index (idempotent), insert_many(ordered=False) —
# duplicate-key errors on re-run are expected and skipped.

import datetime as dt
from pymongo import MongoClient
from pymongo.errors import BulkWriteError

user = dbutils.secrets.get("fvr_dev", "atlas_user")
pw = dbutils.secrets.get("fvr_dev", "atlas_password")
host = dbutils.secrets.get("fvr_dev", "atlas_srv_host")
db = MongoClient(f"mongodb+srv://{user}:{pw}@{host}/fvr?retryWrites=true&w=majority",
                 serverSelectionTimeoutMS=15000)["fvr"]
NOW = dt.datetime.utcnow()

def js(required, props):
    return {"bsonType": "object", "required": required, "properties": props}

def ensure(name, validator):
    if name not in db.list_collection_names():
        db.command("create", name, validator={"$jsonSchema": validator},
                   validationLevel="strict", validationAction="error")

def seed(coll, docs):
    try:
        return len(db[coll].insert_many(docs, ordered=False).inserted_ids)
    except BulkWriteError as e:
        errs = e.details.get("writeErrors", [])
        if any(we["code"] != 11000 for we in errs):
            raise
        return len(docs) - len(errs)          # dups skipped — idempotent re-run

# ============================== STEP 1 — registry ==============================
ensure("users", js(["user_id","email","display_name","roles","is_active","created_at"], {
  "user_id":{"bsonType":"string","pattern":"^usr_"}, "email":{"bsonType":"string","minLength":5},
  "display_name":{"bsonType":"string","minLength":1},
  "roles":{"bsonType":"array","minItems":1,"items":{"enum":["UPLOADER","OPS","ADMIN"]}},
  "department":{"bsonType":["string","null"]}, "is_active":{"bsonType":"bool"},
  "created_at":{"bsonType":"date"}, "last_login_at":{"bsonType":["date","null"]}}))
db.users.create_index([("user_id",1)], unique=True); db.users.create_index([("email",1)], unique=True)

ensure("source_systems", js(["source_id","name","source_type","connection_ref","poll_or_push","is_active"], {
  "source_id":{"bsonType":"string","pattern":"^src_"}, "name":{"bsonType":"string","minLength":1},
  "source_type":{"enum":["PORTAL","SHAREPOINT","SFTP","DB_VOLUME"]},
  "connection_ref":{"bsonType":"string","minLength":1}, "poll_or_push":{"enum":["PUSH","POLL"]},
  "detection_config":{"bsonType":["object","null"]}, "is_active":{"bsonType":"bool"}}))
db.source_systems.create_index([("source_id",1)], unique=True)

ensure("file_types", js(["file_type_id","code","allowed_extensions","header_row","is_active","version"], {
  "file_type_id":{"bsonType":"string","pattern":"^ft_"}, "code":{"bsonType":"string","minLength":2},
  "description":{"bsonType":["string","null"]},
  "allowed_extensions":{"bsonType":"array","minItems":1,"items":{"bsonType":"string","pattern":"^\\."}},
  "max_file_size_mb":{"bsonType":["int","null"],"minimum":1}, "min_rows":{"bsonType":["int","null"],"minimum":0},
  "encoding":{"bsonType":["string","null"]}, "delimiter":{"bsonType":["string","null"]},
  "header_row":{"bsonType":"bool"}, "identification":{"bsonType":["object","null"]},
  "is_active":{"bsonType":"bool"}, "version":{"bsonType":"int","minimum":1}}))
db.file_types.create_index([("file_type_id",1)], unique=True); db.file_types.create_index([("code",1)], unique=True)

FIELD = {"bsonType":"object","required":["order","name","data_type","nullable"],"properties":{
  "order":{"bsonType":"int","minimum":1}, "name":{"bsonType":"string","minLength":1},
  "data_type":{"enum":["string","integer","decimal","date","boolean","timestamp"]},
  "nullable":{"bsonType":"bool"}, "max_length":{"bsonType":["int","null"]},
  "format":{"bsonType":["string","null"]}, "is_pii":{"bsonType":"bool"},
  "pii_classification":{"bsonType":["string","null"]}}}
ensure("schema_versions", js(["schema_id","file_type_id","version","status","effective_from","fields","created_by","created_at"], {
  "schema_id":{"bsonType":"string","pattern":"^sch_"}, "file_type_id":{"bsonType":"string","pattern":"^ft_"},
  "version":{"bsonType":"int","minimum":1}, "status":{"enum":["draft","active","retired"]},
  "effective_from":{"bsonType":"date"}, "fields":{"bsonType":"array","minItems":1,"items":FIELD},
  "created_by":{"bsonType":"string"}, "created_at":{"bsonType":"date"}}))
db.schema_versions.create_index([("schema_id",1)], unique=True)
db.schema_versions.create_index([("file_type_id",1),("version",1)], unique=True)
db.schema_versions.create_index([("file_type_id",1)], unique=True, partialFilterExpression={"status":"active"})

ensure("validation_rules", js(["rule_id","rule_code","category","severity","params","priority","is_active","version","created_by","created_at"], {
  "rule_id":{"bsonType":"string","pattern":"^rule_"}, "rule_code":{"bsonType":"string","minLength":3},
  "category":{"enum":["FILE","HEADER","CONTENT"]}, "severity":{"enum":["ERROR","WARN"]},
  "applies_to":{"bsonType":["string","null"]}, "params":{"bsonType":"object"},
  "priority":{"bsonType":"int","minimum":1}, "is_active":{"bsonType":"bool"},
  "version":{"bsonType":"int","minimum":1}, "created_by":{"bsonType":"string"},
  "created_at":{"bsonType":"date"}, "updated_at":{"bsonType":["date","null"]}}))
db.validation_rules.create_index([("rule_id",1)], unique=True)
db.validation_rules.create_index([("rule_code",1),("version",1)], unique=True)

ensure("routing_policies", js(["policy_id","name","priority","match","action","is_active"], {
  "policy_id":{"bsonType":"string","pattern":"^pol_"}, "name":{"bsonType":"string","minLength":1},
  "priority":{"bsonType":"int","minimum":1}, "match":{"bsonType":"object"},
  "action":{"enum":["MOVE_INTAKE","MOVE_QUARANTINE","REJECT","NOTIFY_ONLY"]}, "is_active":{"bsonType":"bool"}}))
db.routing_policies.create_index([("policy_id",1)], unique=True)
db.routing_policies.create_index([("priority",1)])

ensure("volume_map", js(["map_id","file_type_id","destination_type","volume_root","subfolder","naming_convention"], {
  "map_id":{"bsonType":"string","pattern":"^vol_"}, "file_type_id":{"bsonType":"string"},
  "destination_type":{"enum":["STAGING","INTAKE","QUARANTINE","REJECT"]},
  "volume_root":{"bsonType":"string","minLength":1}, "subfolder":{"bsonType":"string"},
  "naming_convention":{"bsonType":"string","minLength":1}}))
db.volume_map.create_index([("map_id",1)], unique=True)
db.volume_map.create_index([("file_type_id",1),("destination_type",1)], unique=True)

# ---------- Step 1 seeds (real mdd_platform-derived contracts) ----------
seed("source_systems", [{"source_id":"src_portal","name":"UI Portal (React)","source_type":"PORTAL",
  "connection_ref":"internal:portal","poll_or_push":"PUSH","detection_config":{},
  "is_active":True,"created_at":NOW}])

FT = lambda ftid, code, desc, pat: {"file_type_id":ftid,"code":code,"description":desc,
  "allowed_extensions":[".csv"],"max_file_size_mb":500,"min_rows":1,"encoding":"utf-8",
  "delimiter":",","header_row":True,"identification":{"patterns":[pat],"header_fingerprint":[]},
  "is_active":True,"version":1}
seed("file_types", [
  FT("ft_customer","customer","Customer master", r"^cust(omer)?[_-].*\.csv$"),
  FT("ft_order","order","Orders", r"^orders?[_-].*\.csv$"),
  FT("ft_payments","payments","Payments (MDD billing domain)", r"^payments?[_-].*\.csv$"),
  FT("ft_product","product","Product catalog", r"^product[s]?[_-].*\.csv$"),
  FT("ft_employee","employee","Employee roster", r"^employee[s]?[_-].*\.csv$")])

def F(defs):
    out = []
    for i, d in enumerate(defs):
        pii = d[3] if len(d) > 3 else None
        out.append({"order":i+1,"name":d[0],"data_type":d[1],"nullable":d[2],
                    "max_length":None,"format":None,"is_pii":bool(pii),"pii_classification":pii})
    return out
def SCH(sid, ft, defs):
    return {"schema_id":sid,"file_type_id":ft,"version":1,"status":"active","effective_from":NOW,
            "fields":F(defs),"created_by":"system:seed","created_at":NOW}
seed("schema_versions", [
  SCH("sch_customer_v1","ft_customer",[["customer_id","string",False],["first_name","string",False,"PII"],
    ["last_name","string",False,"PII"],["email","string",True,"PII"],["phone","string",True,"PII"],
    ["address","string",True,"PII"],["city","string",True],["country","string",True],
    ["signup_date","date",False],["loyalty_tier","string",True]]),
  SCH("sch_order_v1","ft_order",[["order_id","string",False],["customer_id","string",False],
    ["order_date","date",False],["amount","decimal",False],["currency","string",False],
    ["status","string",True],["channel","string",True]]),
  SCH("sch_payments_v1","ft_payments",[["payment_id","string",False],["order_id","string",False],
    ["payment_method","string",False],["amount","decimal",False],["card_last4","string",True,"PII"],
    ["payment_date","date",False],["status","string",True]]),
  SCH("sch_product_v1","ft_product",[["product_id","string",False],["product_name","string",False],
    ["category","string",True],["price","decimal",False],["stock_qty","integer",False],
    ["supplier_id","string",True],["launch_date","date",True]]),
  SCH("sch_employee_v1","ft_employee",[["employee_id","string",False],["full_name","string",False,"PII"],
    ["email","string",True,"PII"],["phone","string",True,"PII"],["department","string",True],
    ["hire_date","date",True],["salary","decimal",True,"PII"]])])

def RULE(rid, code, cat, sev, prio, params, applies=None):
    return {"rule_id":rid,"rule_code":code,"category":cat,"severity":sev,"applies_to":applies,
            "params":params,"priority":prio,"is_active":True,"version":1,
            "created_by":"system:seed","created_at":NOW}
seed("validation_rules", [
  RULE("rule_file_type_allowed","FILE_TYPE_ALLOWED","FILE","ERROR",10,{"enforce_declared_extension":True}),
  RULE("rule_file_size_max","FILE_SIZE_MAX","FILE","ERROR",20,{"max_mb":500}),
  RULE("rule_file_not_empty","FILE_NOT_EMPTY","FILE","ERROR",30,{"min_bytes":1}),
  RULE("rule_file_not_duplicate","FILE_NOT_DUPLICATE","FILE","ERROR",40,{"scope":"global"}),
  RULE("rule_header_required","HEADER_REQUIRED","HEADER","ERROR",10,{}),
  RULE("rule_schema_match","SCHEMA_MATCH","HEADER","ERROR",20,{"mode":"exact_names"}),
  RULE("rule_column_order_match","COLUMN_ORDER_MATCH","HEADER","ERROR",30,{"mode":"strict"}),
  RULE("rule_file_not_empty_rows","FILE_NOT_EMPTY_ROWS","CONTENT","WARN",10,{"min_rows":1})])

seed("routing_policies", [
  {"policy_id":"pol_error_quarantine","name":"Any ERROR verdict → Quarantine","priority":10,
   "match":{"any_error":True},"action":"MOVE_QUARANTINE","is_active":True},
  {"policy_id":"pol_unidentified_quarantine","name":"Unidentified → Quarantine","priority":20,
   "match":{"status":"UNIDENTIFIED"},"action":"MOVE_QUARANTINE","is_active":True},
  {"policy_id":"pol_passed_intake","name":"PASSED/WARNED → Intake","priority":30,
   "match":{"status":{"$in":["PASSED","WARNED"]}},"action":"MOVE_INTAKE","is_active":True}])

INTAKE, QUAR, STAGING = ("/Volumes/mdd_platform/fvr/intake", "/Volumes/mdd_platform/fvr/quarantine",
                        "/Volumes/mdd_platform/fvr/staging")
seed("volume_map", [
  {"map_id":"vol_customer_intake","file_type_id":"ft_customer","destination_type":"INTAKE","volume_root":INTAKE,"subfolder":"customer","naming_convention":"{file_type}_{file_id}_{ts}.ext"},
  {"map_id":"vol_order_intake","file_type_id":"ft_order","destination_type":"INTAKE","volume_root":INTAKE,"subfolder":"order","naming_convention":"{file_type}_{file_id}_{ts}.ext"},
  {"map_id":"vol_payments_intake","file_type_id":"ft_payments","destination_type":"INTAKE","volume_root":INTAKE,"subfolder":"payments","naming_convention":"{file_type}_{file_id}_{ts}.ext"},
  {"map_id":"vol_product_intake","file_type_id":"ft_product","destination_type":"INTAKE","volume_root":INTAKE,"subfolder":"product","naming_convention":"{file_type}_{file_id}_{ts}.ext"},
  {"map_id":"vol_employee_intake","file_type_id":"ft_employee","destination_type":"INTAKE","volume_root":INTAKE,"subfolder":"employee","naming_convention":"{file_type}_{file_id}_{ts}.ext"},
  {"map_id":"vol_catchall_quarantine","file_type_id":"*","destination_type":"QUARANTINE","volume_root":QUAR,"subfolder":"unidentified","naming_convention":"{file_type}_{file_id}_{ts}.ext"},
  {"map_id":"vol_staging_catchall","file_type_id":"*","destination_type":"STAGING","volume_root":STAGING,"subfolder":"sessions","naming_convention":"{original_name}"}])

# users bootstrap admin (Step-1 // EDIT cell) intentionally NOT seeded — set via portal or mongosh.

# ============================ STEP 2 — operational =============================
ensure("event_log", js(["event_id","event_type","source_id","detection_method","dedup_key",
                        "payload","processing_status","retry_count","received_at"], {
  "event_id":{"bsonType":"string","pattern":"^evt_"},
  "event_type":{"enum":["FILE_ARRIVED","FILE_STAGED","FILE_IDENTIFIED","FILE_UNIDENTIFIED",
    "VALIDATION_COMPLETED","FILE_ROUTED","TRANSFER_COMPLETED","TRANSFER_FAILED",
    "BRONZE_PROCESSED","RECONCILIATION_COMPLETED","FILE_ARCHIVED","FILE_PURGED"]},
  "source_id":{"bsonType":"string"}, "detection_method":{"enum":["PUSH","WEBHOOK","WATCHER","POLL"]},
  "dedup_key":{"bsonType":"string","minLength":1}, "payload":{"bsonType":"object"},
  "processing_status":{"enum":["RECEIVED","PROCESSING","PROCESSED","DEAD"]},
  "retry_count":{"bsonType":"int","minimum":0}, "last_error":{"bsonType":["string","null"]},
  "correlation_id":{"bsonType":["string","null"]}, "received_at":{"bsonType":"date"}}))
db.event_log.create_index([("event_id",1)], unique=True)
db.event_log.create_index([("event_type",1),("dedup_key",1)], unique=True)
db.event_log.create_index([("processing_status",1),("received_at",1)])
db.event_log.create_index([("received_at",1)], partialFilterExpression={"processing_status":"DEAD"})
# TTL (30d, PROCESSED only) intentionally ABSENT pending compliance sign-off

PART = {"bsonType":"object","required":["file_id","status","bytes_received"],"properties":{
  "file_id":{"bsonType":"string"}, "staged_path":{"bsonType":["string","null"]},
  "status":{"enum":["UPLOADING","FINALIZED","ABANDONED"]},
  "bytes_received":{"bsonType":"long","minimum":0},
  "sha256":{"bsonType":["string","null"],"maxLength":64}}}
ensure("upload_sessions", js(["session_id","source_id","user_id","user_email","client_ip",
                              "roles_snapshot","files_received_count","parts","status","started_at"], {
  "session_id":{"bsonType":"string","pattern":"^ses_"}, "source_id":{"bsonType":"string"},
  "status":{"enum":["IN_PROGRESS","COMPLETED","COMPLETED_PARTIAL","INCOMPLETE","ABANDONED"]},
  "user_id":{"bsonType":"string"}, "user_email":{"bsonType":"string"}, "client_ip":{"bsonType":"string"},
  "machine_name":{"bsonType":["string","null"]}, "user_agent":{"bsonType":["string","null"]},
  "auth_method":{"bsonType":["string","null"]},
  "roles_snapshot":{"bsonType":"array","minItems":1,"items":{"enum":["UPLOADER","OPS","ADMIN"]}},
  "files_declared_count":{"bsonType":["int","null"],"minimum":0},
  "files_received_count":{"bsonType":"int","minimum":0}, "parts":{"bsonType":"array","items":PART},
  "started_at":{"bsonType":"date"}, "completed_at":{"bsonType":["date","null"]}}))
db.upload_sessions.create_index([("session_id",1)], unique=True)
db.upload_sessions.create_index([("user_id",1),("started_at",-1)])
db.upload_sessions.create_index([("status",1),("started_at",1)])

ensure("file_registry", js(["file_id","source_id","original_name","size_bytes","extension",
                            "sha256_checksum","current_status","retry_count","received_at","updated_at"], {
  "file_id":{"bsonType":"string","pattern":"^file_"}, "source_id":{"bsonType":"string"},
  "original_name":{"bsonType":"string","minLength":1},
  "staged_path":{"bsonType":["string","null"]}, "gridfs_file_id":{"bsonType":["objectId","null"]},
  "size_bytes":{"bsonType":["long","int"],"minimum":0}, "extension":{"bsonType":"string"},
  "mime_type":{"bsonType":["string","null"]}, "sha256_checksum":{"bsonType":"string","minLength":64,"maxLength":64},
  "file_type_id":{"bsonType":["string","null"]}, "schema_version_id":{"bsonType":["string","null"]},
  "identification_method":{"enum":["EXPLICIT","FILENAME_PATTERN","HEADER_FINGERPRINT","CONTENT_SNIFF",None]},
  "identification_confidence":{"bsonType":["double","int","null"]},
  "identification_evidence":{"bsonType":["object","null"]}, "unidentified_reason":{"bsonType":["string","null"]},
  "upload_session_id":{"bsonType":["string","null"]}, "uploaded_by":{"bsonType":["string","null"]},
  "client_ip":{"bsonType":["string","null"]}, "machine_name":{"bsonType":["string","null"]},
  "current_status":{"enum":["RECEIVED","IDENTIFYING","VALIDATING","PASSED","FAILED","WARNED",
    "UNIDENTIFIED","ROUTED_INTAKE","ROUTED_QUARANTINE","REJECTED","FRAMEWORK_ERROR"]},
  "batch_id":{"bsonType":["string","null"]}, "retry_count":{"bsonType":"int","minimum":0},
  "received_at":{"bsonType":"date"}, "updated_at":{"bsonType":"date"},
  "volume_purged_at":{"bsonType":["date","null"]}, "legal_hold":{"bsonType":["bool","null"]}}))
db.file_registry.create_index([("file_id",1)], unique=True)
db.file_registry.create_index([("sha256_checksum",1)], unique=True)
db.file_registry.create_index([("current_status",1)])
db.file_registry.create_index([("source_id",1),("received_at",-1)])
db.file_registry.create_index([("upload_session_id",1)])
db.file_registry.create_index([("file_type_id",1),("current_status",1),("received_at",1)],
                               partialFilterExpression={"volume_purged_at":None})

ensure("validation_results", js(["result_id","file_id","run_seq","rule_id","rule_version",
                                 "status","details","executed_at"], {
  "result_id":{"bsonType":"string","pattern":"^vr_"}, "file_id":{"bsonType":"string"},
  "batch_id":{"bsonType":["string","null"]}, "run_seq":{"bsonType":"int","minimum":1},
  "rule_id":{"bsonType":"string"}, "rule_version":{"bsonType":"int","minimum":1},
  "status":{"enum":["PASS","FAIL","WARN","SKIPPED","ERROR"]}, "details":{"bsonType":"object"},
  "executed_at":{"bsonType":"date"}, "duration_ms":{"bsonType":["int","long","null"],"minimum":0}}))
db.validation_results.create_index([("result_id",1)], unique=True)
db.validation_results.create_index([("file_id",1),("run_seq",1),("executed_at",-1)])

ensure("file_transfers", js(["transfer_id","file_id","from_location","to_location",
                             "direction","status","attempts","source_sha256"], {
  "transfer_id":{"bsonType":"string","pattern":"^trf_"}, "file_id":{"bsonType":"string"},
  "policy_id":{"bsonType":["string","null"]}, "from_location":{"bsonType":"string"},
  "to_location":{"bsonType":"string"},
  "direction":{"enum":["MONGO_TO_INTAKE","MONGO_TO_QUARANTINE","MONGO_TO_ARCHIVE"]},  # legacy labels — see header note
  "status":{"enum":["PENDING","IN_PROGRESS","COMPLETED","FAILED","RETRYING"]},
  "attempts":{"bsonType":"int","minimum":0}, "last_error":{"bsonType":["string","null"]},
  "source_sha256":{"bsonType":"string","minLength":64,"maxLength":64},
  "dest_sha256":{"bsonType":["string","null"],"minLength":64,"maxLength":64},
  "bytes_written":{"bsonType":["long","int","null"],"minimum":0}, "worker_id":{"bsonType":["string","null"]},
  "started_at":{"bsonType":["date","null"]}, "completed_at":{"bsonType":["date","null"]}}))
db.file_transfers.create_index([("transfer_id",1)], unique=True)
db.file_transfers.create_index([("file_id",1),("direction",1)], unique=True,
                               partialFilterExpression={"status":"COMPLETED"})
db.file_transfers.create_index([("status",1),("started_at",1)])

ensure("file_lifecycle", js(["event_id","file_id","location","timestamp"], {
  "event_id":{"bsonType":"string","pattern":"^flc_"}, "file_id":{"bsonType":"string"},
  "location":{"enum":["STAGING_VOLUME","GRIDFS","INTAKE_VOLUME","QUARANTINE","ARCHIVE","PURGED"]},
  "timestamp":{"bsonType":"date"}, "triggered_by_job":{"bsonType":["string","null"]}}))
db.file_lifecycle.create_index([("event_id",1)], unique=True)
db.file_lifecycle.create_index([("file_id",1),("timestamp",1)])

ensure("audit_log", js(["event_id","ts","actor","entity_type","entity_id","action"], {
  "event_id":{"bsonType":"string","pattern":"^aud_"}, "ts":{"bsonType":"date"},
  "actor":{"bsonType":"string","minLength":1}, "entity_type":{"bsonType":"string"},
  "entity_id":{"bsonType":"string"},
  "action":{"enum":["CONFIG_CHANGE","STATUS_CHANGE","FILE_ROUTED","RULE_ADDED","FILE_PURGED","OTHER"]},
  "before":{"bsonType":["object","null"]}, "after":{"bsonType":["object","null"]},
  "approver":{"bsonType":["string","null"]}, "correlation_id":{"bsonType":["string","null"]}}))
db.audit_log.create_index([("event_id",1)], unique=True)
db.audit_log.create_index([("entity_id",1),("ts",-1)])
db.audit_log.create_index([("ts",1)])

ensure("notifications", js(["notification_id","event","channel","recipients","is_active"], {
  "notification_id":{"bsonType":"string","pattern":"^not_"}, "event":{"bsonType":"string","minLength":1},
  "condition":{"bsonType":["object","null"]}, "channel":{"enum":["email","teams","webhook"]},
  "recipients":{"bsonType":"array","minItems":1,"items":{"bsonType":"string"}},
  "is_active":{"bsonType":"bool"}}))
db.notifications.create_index([("notification_id",1)], unique=True)
db.notifications.create_index([("event",1),("is_active",1)])

seed("notifications", [            # recipients are placeholders — open item, edit in Mongo
  {"notification_id":"not_transfer_failed","event":"TRANSFER_FAILED","condition":{"min_attempts":3},
   "channel":"email","recipients":["data-ops@yourcompany.com"],"is_active":True},
  {"notification_id":"not_framework_error","event":"FRAMEWORK_ERROR","condition":{"min_retry_count":3},
   "channel":"email","recipients":["data-ops@yourcompany.com"],"is_active":True},
  {"notification_id":"not_session_incomplete","event":"SESSION_INCOMPLETE","condition":{},
   "channel":"email","recipients":["portal-support@yourcompany.com"],"is_active":True}])

# =========================== VERIFICATION (Steps 1-2) ==========================
EXPECTED = {"source_systems":1,"file_types":5,"schema_versions":5,"validation_rules":8,
            "routing_policies":3,"volume_map":7,"users":0,
            "event_log":0,"upload_sessions":0,"file_registry":0,"validation_results":0,
            "file_transfers":0,"file_lifecycle":0,"audit_log":0,"notifications":3}
problems = []
for c, n in EXPECTED.items():
    if c not in db.list_collection_names():
        problems.append(f"{c}: MISSING"); continue
    actual = db[c].count_documents({})
    if actual != n: problems.append(f"{c}: expected {n} docs, found {actual}")
    info = db.command("listCollections", filter={"name": c})["cursor"]["firstBatch"]
    va = (info[0].get("options") or {}).get("validationAction") if info else None
    if va != "error": problems.append(f"{c}: validationAction={va}")
ttl = [i for i in db.event_log.index_information().values() if i.get("expireAfterSeconds")]
if ttl: problems.append(f"event_log TTL indexes present: {len(ttl)} (should be 0 until compliance)")
viol = list(db.schema_versions.aggregate([
    {"$match":{"status":"active"}}, {"$group":{"_id":"$file_type_id","n":{"$sum":1}}},
    {"$match":{"n":{"$ne":1}}}]))
if viol: problems.append(f"active-schema violations: {viol}")
for c, g in {"event_log":"event_type_1_dedup_key_1", "file_registry":"sha256_checksum_1",
             "file_transfers":"file_id_1_direction_1", "schema_versions":"file_type_id_1"}.items():
    if g not in db[c].index_information(): problems.append(f"{c}: unique guard {g} missing")
print("STEPS 1-2 APPLY + VERIFY:", "PASS — 15 collections, seeds 1/5/5/8/3/7 + notifications 3, all guards present" if not problems else "FAIL")
for p in problems: print("  !!", p)
assert not problems, problems